In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [4]:
df = pd.read_excel('../data/online_retail_II.xlsx', sheet_name='online_retail_II')

In [5]:
print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
df.head(3)

Total rows: 1,048,575
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

First 3 rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


In [7]:
import duckdb

con = duckdb.connect()
con.register('orders', df)

profile = con.execute("""
SELECT
  COUNT(*)                                    AS total_rows,
  COUNT(DISTINCT Invoice)                     AS unique_invoices,
  COUNT(DISTINCT "Customer ID")               AS unique_customers,
  COUNT(DISTINCT StockCode)                   AS unique_products,
  COUNT(DISTINCT Country)                     AS unique_countries,
  MIN(InvoiceDate)                            AS earliest_date,
  MAX(InvoiceDate)                            AS latest_date,
  ROUND(100.0 * SUM(CASE WHEN "Customer ID" IS NULL
    THEN 1 ELSE 0 END) / COUNT(*), 1)         AS pct_null_customers,
  ROUND(100.0 * SUM(CASE WHEN Invoice LIKE 'C%'
    THEN 1 ELSE 0 END) / COUNT(*), 1)         AS pct_cancellations
FROM orders
""").df()

print(profile.to_string())

   total_rows  unique_invoices  unique_customers  unique_products  unique_countries       earliest_date         latest_date  pct_null_customers  pct_cancellations
0     1048575            52961              5924             5304                43 2009-12-01 07:45:00 2011-12-04 13:15:00                22.6                1.8


In [5]:
import sys
!{sys.executable} -m pip install duckdb

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   --------------------- ------------------ 7.1/13.1 MB 44.4 MB/s eta 0:00:01
   ---------------------------------------  13.1/13.1 MB 46.1 MB/s eta 0:00:01
   ---------------------------------------  13.1/13.1 MB 46.1 MB/s eta 0:00:01
   ---------------------------------------- 13.1/13.1 MB 16.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# ── Step 1: Remove cancellations (invoices starting with 'C')
cancellations = df[df['Invoice'].astype(str).str.startswith('C')]
print(f"Cancelled orders removed: {len(cancellations):,}")
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# ── Step 2: Remove rows with null Customer ID
null_customers = df['Customer ID'].isna().sum()
print(f"Null Customer ID rows removed: {null_customers:,}")
df = df.dropna(subset=['Customer ID'])

# ── Step 3: Remove negative or zero quantities
bad_qty = df[df['Quantity'] <= 0]
print(f"Non-positive quantity rows removed: {len(bad_qty):,}")
df = df[df['Quantity'] > 0]

# ── Step 4: Remove zero or negative prices
bad_price = df[df['Price'] <= 0]
print(f"Zero/negative price rows removed: {len(bad_price):,}")
df = df[df['Price'] > 0]

# ── Step 5: Create revenue column
df['Revenue'] = df['Quantity'] * df['Price']

# ── Step 6: Fix data types
df['Customer ID'] = df['Customer ID'].astype(int).astype(str)
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')

print(f"\n── Final clean dataset ──")
print(f"Rows remaining:  {len(df):,}")
print(f"Unique customers: {df['Customer ID'].nunique():,}")
print(f"Unique invoices:  {df['Invoice'].nunique():,}")
print(f"Total revenue:    £{df['Revenue'].sum():,.0f}")
print(f"Date range:       {df['InvoiceDate'].min().date()} to {df['InvoiceDate'].max().date()}")

Cancelled orders removed: 19,261
Null Customer ID rows removed: 235,934
Non-positive quantity rows removed: 0
Zero/negative price rows removed: 71

── Final clean dataset ──
Rows remaining:  793,309
Unique customers: 5,860
Unique invoices:  36,457
Total revenue:    £17,324,932
Date range:       2009-12-01 to 2011-12-04
